In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/Users/samantha/QuantUS-Plugins-CEUS/China_Data
/Users/samantha/QuantUS-Plugins-CEUS


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'nifti', 'custom_dicom', 'mp4']


In [167]:
scan_type = 'nifti'

scan_path = '/Users/samantha/Desktop/tul/china data/p11/new_v1/CEUS-24685.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [168]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [169]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [170]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/tul/china data/p11/new_v1/v1_new_necrotic_manual.nii.gz'
seg_loader_kwargs = {}

In [171]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis (Parametric Map Mode)

In [172]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))

Available analysis types: ['curves_paramap', 'curves']


In [173]:
# IMPORTANT: Use curves_paramap for parametric map generation
analysis_type = 'curves_paramap'

print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis functions: ['pyradiomics', 'tic']


In [174]:
analysis_funcs = ['tic']

required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: ['cor_vox_len', 'sag_vox_ovrlp', 'cor_vox_ovrlp', 'sag_vox_len', 'ax_vox_len', 'ax_vox_ovrlp']


In [175]:
# Set frame rate
image_data.frame_rate = 1

# Required kwargs for parametric map analysis
analysis_kwargs = {
    'ax_vox_ovrlp': 50,
    'sag_vox_ovrlp': 50,
    'cor_vox_ovrlp': 50,
    'ax_vox_len': 5,
    'sag_vox_len': 5,
    'cor_vox_len': 5,
}

In [176]:
import copy
import numpy as np
from tqdm import tqdm
from src.time_series_analysis.curves_paramap.framework import CurvesParamapAnalysis


class VectorizedCurvesParamapAnalysis(CurvesParamapAnalysis):

    def compute_curves(self):
        data = self.image_data.intensities_for_analysis
        is_3d = data.ndim == 4
        if not is_3d and data.ndim != 3:
            raise ValueError('Image data must be either 2D+time or 3D+time.')

        n_frames = data.shape[3] if is_3d else data.shape[0]

        self.curves = []
        for ix, window in tqdm(enumerate(self.windows), desc='Computing curves', total=len(self.windows)):
            entry = {}
            if is_3d:
                ax_start, sag_start, cor_start, ax_end, sag_end, cor_end = window
                entry['Window-Axial Start Pix'] = ax_start
                entry['Window-Sagittal Start Pix'] = sag_start
                entry['Window-Coronal Start Pix'] = cor_start
                entry['Window-Axial End Pix'] = ax_end
                entry['Window-Sagittal End Pix'] = sag_end
                entry['Window-Coronal End Pix'] = cor_end
                window_data = data[sag_start:sag_end+1, cor_start:cor_end+1, ax_start:ax_end+1, :]
                means = window_data.reshape(-1, n_frames).mean(axis=0)
            else:
                ax_start, sag_start, ax_end, sag_end = window
                entry['Window-Axial Start Pix'] = ax_start
                entry['Window-Sagittal Start Pix'] = sag_start
                entry['Window-Axial End Pix'] = ax_end
                entry['Window-Sagittal End Pix'] = sag_end
                window_data = data[:, ax_start:ax_end+1, sag_start:sag_end+1]
                means = window_data.reshape(n_frames, -1).mean(axis=1)

            entry['TIC'] = means.tolist()
            self.curves.append(entry)

        if self.curves_output_path:
            self.save_curves()


print('VectorizedCurvesParamapAnalysis defined')

VectorizedCurvesParamapAnalysis defined


In [177]:
analyzed_image_data = copy.deepcopy(image_data)

analysis_obj = VectorizedCurvesParamapAnalysis(analyzed_image_data, seg_data, analysis_funcs, **analysis_kwargs)
analysis_obj.compute_curves()

print("Analysis object type:", type(analysis_obj))

Computing curves: 100%|██████████| 95754/95754 [00:06<00:00, 15425.31it/s]

Analysis object type: <class '__main__.VectorizedCurvesParamapAnalysis'>


## Curve Quantification

In [178]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['auc_no_fit', 'cmus_firstorder', 'dte', 'first_order_full', 'first_order_select', 'lognormal_fit_full', 'lognormal_fit_select', 'wash_rates'])


In [179]:
function_names = ['lognormal_fit_full']  # or [] for all functions
output_path = '/Users/samantha/Desktop/tul/china data/p11/new_v1/necrotic_paramap/output.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
    'tic_name': 'TIC'
}

In [180]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

# Verify analysis type
print("curve_quant.analysis_objs type:", type(curve_quant.analysis_objs))

Curve is constant, cannot normalize.
curve_quant.analysis_objs type: <class '__main__.VectorizedCurvesParamapAnalysis'>


## Parametric Map Saving

In [181]:
from src.entrypoints import visualization_step

vis_type = 'paramap'
params = []
vis_funcs = []
vis_kwargs = {
    'paramap_folder_path': '/Users/samantha/Desktop/tul/china data/p11/new_v1/necrotic_paramap',
    'hide_all_visualizations': False,
}

vis_obj = visualization_step(curve_quant, vis_type, params, vis_funcs, **vis_kwargs)